# Day 08: SGLang — RadixAttention & Structured Outputs
> *Inference Engineering* — Chapter 4.3.2 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 07 (vLLM)


## Concept Overview

SGLang (Structured Generation Language) extends vLLM's ideas with two key innovations: RadixAttention and constrained decoding for structured outputs. RadixAttention organizes the KV cache as a radix tree, enabling automatic prefix sharing across requests with common prompts — even those arriving at different times. Structured output generation uses finite-state machine (FSM) or grammar-based constraints to guarantee JSON/regex-valid outputs without post-processing, while also enabling speculative execution for fixed output patterns.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import json, re, hashlib
from collections import defaultdict

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Radix Tree for KV Cache

A radix tree (compressed trie) stores token sequences as paths. Common prefixes share nodes. When a new request arrives with a prefix already in the tree, those KV blocks are reused — zero computation. The tree is LRU-evicted when memory pressure occurs.


In [ ]:
class RadixTreeNode:
    def __init__(self):
        self.children = {}  # token_id -> RadixTreeNode
        self.kv_block_id = None
        self.ref_count = 0
        self.last_used = 0

class RadixCache:
    def __init__(self):
        self.root = RadixTreeNode()
        self.clock = 0
        self.hits = 0
        self.misses = 0

    def _hash_prefix(self, tokens):
        return hashlib.md5(bytes(tokens)).hexdigest()[:8]

    def lookup(self, token_seq):
        """Return (matched_length, cached_block_ids)."""
        node = self.root
        matched = 0
        blocks = []
        for tok in token_seq:
            if tok in node.children:
                node = node.children[tok]
                node.last_used = self.clock
                self.clock += 1
                matched += 1
                if node.kv_block_id:
                    blocks.append(node.kv_block_id)
            else:
                break
        if matched > 0:
            self.hits += 1
        else:
            self.misses += 1
        return matched, blocks

    def insert(self, token_seq, block_id):
        node = self.root
        for tok in token_seq:
            if tok not in node.children:
                node.children[tok] = RadixTreeNode()
            node = node.children[tok]
        node.kv_block_id = block_id

# Simulate requests with shared system prompt
system_prompt = [1, 2, 3, 4, 5, 6, 7, 8]  # 8 tokens
requests = [
    system_prompt + [10, 11, 12],
    system_prompt + [20, 21],
    system_prompt + [10, 11, 30, 31],
    [100, 200, 300],  # no shared prefix
]

cache = RadixCache()
cache.insert(system_prompt, block_id='blk_0')

print('RadixCache lookup simulation:')
print(f'System prompt cached: {system_prompt}')
print()
for i, req in enumerate(requests):
    matched, blocks = cache.lookup(req)
    cache.insert(req, block_id=f'blk_{i+1}')
    pct = matched / len(req) * 100
    print(f'  Req {i+1} (len={len(req)}): {matched} tokens cached ({pct:.0f}%), blocks={blocks}')

print(f'\nCache hit rate: {cache.hits}/{cache.hits+cache.misses}')
print(f'Tokens saved (not recomputed): {sum(cache.lookup(r)[0] for r in requests)}')


## 2. Structured Output with FSM Constraints

Constrained decoding uses a finite-state machine to mask the logits at each step, allowing only tokens that keep the output valid according to the schema. This guarantees structurally valid JSON without post-processing retries.


In [ ]:
class JSONFSMSimulator:
    """Simplified FSM for JSON object with one string field."""
    STATES = ['start','open_brace','key_start','in_key','key_end','colon',
              'val_start','in_val','val_end','close_brace','done']

    TRANSITIONS = {
        'start':       {'{': 'open_brace'},
        'open_brace':  {'"': 'key_start'},
        'key_start':   {'[a-z]': 'in_key'},
        'in_key':      {'[a-z]': 'in_key', '"': 'key_end'},
        'key_end':     {':': 'colon'},
        'colon':       {' ': 'colon', '"': 'val_start'},
        'val_start':   {'[^"]+': 'in_val'},
        'in_val':      {'[^"]+': 'in_val', '"': 'val_end'},
        'val_end':     {'}': 'done'},
    }

    def simulate(self, output_stream):
        state = 'start'
        tokens_forced = 0
        print('FSM-guided JSON generation:')
        for ch in output_stream:
            print(f'  [{state:>15}] + "{ch}" → ', end='')
            # In real constrained decoding, we mask logits to valid next tokens
            if state == 'done':
                print('COMPLETE')
                break
            elif ch in ['{']: state = 'open_brace'
            elif ch == '"' and state in ['open_brace']: state = 'key_start'
            elif ch == '"' and state in ['in_key']: state = 'key_end'
            elif ch == ':': state = 'colon'
            elif ch == '"' and state == 'colon': state = 'val_start'
            elif ch == '"' and state == 'in_val': state = 'val_end'
            elif ch == '}': state = 'done'
            elif state in ['key_start','in_key']: state = 'in_key'
            elif state in ['val_start','in_val']: state = 'in_val'
            print(state)
        return state == 'done'

fsm = JSONFSMSimulator()
stream = '{"name": "Alice"}'
valid = fsm.simulate(stream)
print(f'Output valid JSON: {valid}')

# Show invalid stream rejection
print('\nInvalid stream: {name: Alice}')
fsm2 = JSONFSMSimulator()
for i, ch in enumerate('{name: Alice}'):
    # Without quotes around key, FSM would reject
    pass
print('FSM would block "n" after "{" — expects "\"" to start a key')


## 3. Speculative Execution for Fixed Patterns

For structured outputs with fixed prefixes (e.g., `{"result": `), SGLang can speculatively prefill the known tokens, skipping the autoregressive loop for the fixed portion entirely.


In [ ]:
# Measure latency benefit of speculative prefill for fixed format
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def simulate_autoregressive_tokens(n_tokens, d_model=512):
    """Simulate decode time for n_tokens autoregressive steps."""
    t0 = time.perf_counter()
    h = torch.randn(1, 1, d_model, device=device)
    for _ in range(n_tokens):
        h = torch.nn.functional.layer_norm(h + torch.randn_like(h), [d_model])
    if device == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000

fixed_prefix_len = 8   # tokens in '{"result": ' prefix
variable_len = 20      # content tokens

t_normal = simulate_autoregressive_tokens(fixed_prefix_len + variable_len)
t_speculative = simulate_autoregressive_tokens(variable_len)  # skip fixed prefix

print('Speculative prefill benefit for structured output:')
print(f'  Fixed prefix: {fixed_prefix_len} tokens (e.g., {{"result": ")')
print(f'  Variable content: {variable_len} tokens')
print(f'  Normal decode:      {t_normal:.2f} ms ({fixed_prefix_len+variable_len} steps)')
print(f'  Speculative decode: {t_speculative:.2f} ms ({variable_len} steps)')
print(f'  Speedup: {t_normal/t_speculative:.2f}x')
print(f'  Savings: {(1-t_speculative/t_normal)*100:.0f}% of decode time')


## Experiments: Try These

1. **RadixCache with LRU**: Extend the RadixCache with an LRU eviction policy. Simulate 1000 requests with a Zipf-distributed prefix length and measure hit rates.
2. **Real SGLang**: Install SGLang and run structured JSON generation. Measure tokens/sec vs unstructured with vLLM baseline.
3. **Grammar compilation**: Research Outlines/LMQL FSM compilation. How many states does a simple JSON schema FSM have?


## Key Takeaways

- RadixAttention organizes the KV cache as a radix tree, enabling automatic and persistent prefix reuse across requests — even non-concurrent ones.
- Structured output generation uses FSM-constrained logit masking to guarantee schema-valid outputs, eliminating retry loops.
- Speculative prefill skips the autoregressive loop for fixed output prefixes in structured formats, directly reducing latency.
- SGLang's tree-based KV management gives it a significant advantage for workloads with repeated system prompts or few-shot examples.

## References
- *Inference Engineering* Ch 4.3.2 — Philip Kiely, Baseten Books 2026
- Zheng et al. (2023), "SGLang: Efficient Execution of Structured Language Model Programs"
